# Part 1: Bag of Words (词袋模型) - 入门 baseline

本 Notebook 实现 Kaggle Word2Vec 竞赛的 Part 1 教程，使用传统的**词袋模型 (Bag of Words)** + **随机森林分类器**完成情感分类。

## 1. 数据加载

In [ ]:
import pandas as pd
import numpy as np
import os

# 数据路径
DATA_DIR = "../word2vec-nlp-tutorial"
SUBMISSION_DIR = "../submissions"

# 读取训练数据
train = pd.read_csv(
    os.path.join(DATA_DIR, "labeledTrainData.tsv"), 
    header=0, 
    delimiter="\t", 
    quoting=3
)

# 读取测试数据
test = pd.read_csv(
    os.path.join(DATA_DIR, "testData.tsv"), 
    header=0, 
    delimiter="\t", 
    quoting=3
)

print(f"训练集形状：{train.shape}")
print(f"测试集形状：{test.shape}")
print(f"\n训练集列名：{train.columns.tolist()}")
print(f"\n前 3 条训练数据：")
train.head(3)

## 2. 文本清洗

电影评论包含 HTML 标签、标点符号等噪声，需要清洗。

In [ ]:
import re
from bs4 import BeautifulSoup

def review_to_words(raw_review):
    """
    将原始评论转换为干净的单词列表
    
    参数:
        raw_review: 原始评论字符串
    
    返回:
        清洗后的字符串
    """
    # 1. 去除 HTML 标签
    markup = BeautifulSoup(raw_review, 'html.parser')
    markup_text = markup.get_text()
    
    # 2. 去除非字母字符 (保留字母和空格)
    letters_only = re.sub("[^a-zA-Z]", " ", markup_text)
    
    # 3. 转为小写并分割成单词
    words = letters_only.lower().split()
    
    # 4. 返回拼接后的字符串
    return " ".join(words)

# 测试清洗函数
print("原始评论 (前 200 字符):")
print(train['review'][0][:200])
print("\n清洗后评论:")
print(review_to_words(train['review'][0]))

In [ ]:
# 应用清洗到整个数据集
print("正在清洗训练数据...")
clean_train_reviews = []
for i in range(len(train['review'])):
    if (i+1) % 5000 == 0:
        print(f"  已处理 {i+1}/{len(train['review'])} 条")
    clean_train_reviews.append(review_to_words(train['review'][i]))

print("正在清洗测试数据...")
clean_test_reviews = []
for i in range(len(test['review'])):
    if (i+1) % 5000 == 0:
        print(f"  已处理 {i+1}/{len(test['review'])} 条")
    clean_test_reviews.append(review_to_words(test['review'][i]))

print("✓ 数据清洗完成!")

## 3. 词袋模型特征提取

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# 初始化向量化器
vectorizer = CountVectorizer(
    analyzer='word',
    max_features=5000,
    min_df=2,
)

train_data_features = vectorizer.fit_transform(clean_train_reviews).toarray()
test_data_features = vectorizer.transform(clean_test_reviews).toarray()

print(f"训练特征矩阵形状：{train_data_features.shape}")
print(f"测试特征矩阵形状：{test_data_features.shape}")

vocab = vectorizer.get_feature_names_out()
print(f"\n词汇表大小：{len(vocab)}")
print(f"前 20 个词：{vocab[:20]}")

## 4. 训练随机森林分类器

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_val, y_train, y_val = train_test_split(
    train_data_features, 
    train['sentiment'], 
    test_size=0.2, 
    random_state=42
)

print(f"训练集：{X_train.shape}, 验证集：{X_val.shape}")

forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    n_jobs=-1,
    verbose=1
)

print("正在训练随机森林...")
forest.fit(X_train, y_train)
print("✓ 训练完成!")

y_pred = forest.predict(X_val)
val_accuracy = accuracy_score(y_val, y_pred)
print(f"\n验证集准确率：{val_accuracy:.4f} ({val_accuracy*100:.2f}%)")

## 5. 生成并提交结果

In [ ]:
import datetime

forest.fit(train_data_features, train['sentiment'])
result = forest.predict(test_data_features)

output = pd.DataFrame({
    'id': test['id'],
    'sentiment': result
})

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_path = os.path.join(SUBMISSION_DIR, f"bow_baseline_{timestamp}.csv")
output.to_csv(submission_path, index=False, quoting=3)

print(f"✓ 提交文件已保存：{submission_path}")
print(f"  正面 (1): {sum(result)} 条")
print(f"  负面 (0): {len(result) - sum(result)} 条")

## 6. 模型性能分析

In [ ]:
feature_importances = forest.feature_importances_
top_indices = np.argsort(feature_importances)[::-1][:20]

print("Top 20 最重要的单词:")
for i in top_indices:
    print(f"  {vocab[i]}: {feature_importances[i]:.4f}")

## 总结

**本 Part 实现的方法**:
1. ✅ 数据加载 (TSV 格式)
2. ✅ 文本清洗 (去 HTML、标点、转小写)
3. ✅ 词袋模型特征提取 (5000 维)
4. ✅ 随机森林分类器
5. ✅ 生成提交文件

**预期 Kaggle 成绩**: 85-87% 准确率